# AI Reliability Judge — Gemma 4 E2B Fine-tuning

**Track:** Safety & Trust | **Hackathon:** Gemma 4 Good

Fine-tune Gemma 4 E2B (2B params) to judge multi-LLM response reliability risk levels.

**Strategy:** Full-parameter SFT with `trl.SFTTrainer` (no peft/unsloth needed — 2B fits on T4 bf16).

## 1. Environment Setup

In [ ]:
# Install required packages (transformers 5.8.1 should already be available)
!pip install -q trl datasets accelerate kagglehub

In [ ]:
import torch
import kagglehub
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Download Training Data from GitHub

In [ ]:
# Download training data from GitHub repo
!mkdir -p /kaggle/working/data

# Clone from GitHub
!rm -rf /tmp/jinan-drone
!git clone --depth 1 https://github.com/1235789a/jinan-drone.git /tmp/jinan-drone
!cp /tmp/jinan-drone/data/*.jsonl /kaggle/working/data/

# Verify data
!echo "=== Training data ==="
!wc -l /kaggle/working/data/train_chat.jsonl
!echo "=== Validation data ==="
!wc -l /kaggle/working/data/val_chat.jsonl
!echo "=== Sample ==="
!head -1 /kaggle/working/data/train_chat.jsonl | python3 -m json.tool | head -20

## 3. Load Gemma 4 E2B Model

In [ ]:
# Download Gemma 4 E2B via kagglehub
model_path = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e2b-it")
print(f"Model downloaded to: {model_path}")

In [ ]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {model.num_parameters() / 1e9:.2f}B")
print(f"Model dtype: {model.dtype}")

## 4. Prepare Dataset

In [ ]:
# Load datasets
train_dataset = load_dataset("json", data_files="/kaggle/working/data/train_chat.jsonl", split="train")
val_dataset = load_dataset("json", data_files="/kaggle/working/data/val_chat.jsonl", split="train")

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"\nSample keys: {list(train_dataset[0].keys())}")
print(f"Number of messages per sample: {len(train_dataset[0]['messages'])}")

In [ ]:
# Check tokenized lengths to set max_seq_length appropriately
sample_lengths = []
for i in range(min(100, len(train_dataset))):
    text = tokenizer.apply_chat_template(train_dataset[i]["messages"], tokenize=False)
    tokens = tokenizer(text, return_tensors="pt")
    sample_lengths.append(tokens["input_ids"].shape[1])

import statistics
print(f"Token length stats (first {len(sample_lengths)} samples):")
print(f"  Mean: {statistics.mean(sample_lengths):.0f}")
print(f"  Max: {max(sample_lengths)}")
print(f"  Min: {min(sample_lengths)}")
print(f"  Median: {statistics.median(sample_lengths):.0f}")
print(f"  95th percentile: {sorted(sample_lengths)[int(len(sample_lengths)*0.95)]:.0f}")

# Set max_seq_length to cover 95th percentile + some buffer
p95 = sorted(sample_lengths)[int(len(sample_lengths)*0.95)]
MAX_SEQ_LENGTH = min(2048, p95 + 64)
print(f"\nUsing MAX_SEQ_LENGTH = {MAX_SEQ_LENGTH}")

## 5. Training Configuration & Fine-tuning

In [ ]:
# Training configuration optimized for T4 16GB + 2B model
# Full-parameter fine-tuning with gradient checkpointing to fit in memory

training_args = SFTConfig(
    # Output
    output_dir="/kaggle/working/gemma4-reliability-judge",
    
    # Training hyperparameters
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,  # effective batch size = 8
    
    # Learning rate
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    
    # Memory optimization
    gradient_checkpointing=True,
    bf16=True,
    optim="adamw_torch_fused",
    
    # Sequence length
    max_seq_length=MAX_SEQ_LENGTH,
    
    # Logging & Evaluation
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    
    # Misc
    seed=42,
    report_to="none",
    dataloader_pin_memory=False,
)

print(f"Training config:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Grad accum: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  LR: {training_args.learning_rate}")
print(f"  Max seq length: {training_args.max_seq_length}")

In [ ]:
# Initialize SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print("Trainer initialized successfully!")
print(f"Total training steps: ~{len(train_dataset) * training_args.num_train_epochs // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")

In [ ]:
# Start training
print("="*60)
print("Starting fine-tuning...")
print("="*60)
print(f"GPU Memory before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

train_result = trainer.train()

print(f"\n{'='*60}")
print("Training Complete!")
print(f"{'='*60}")
print(f"Total steps: {train_result.global_step}")
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"GPU Memory after training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 6. Save Model & Evaluate

In [ ]:
# Save the fine-tuned model
save_path = "/kaggle/working/gemma4-reliability-judge/final"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved to: {save_path}")

# Evaluate on validation set
eval_results = trainer.evaluate()
print(f"\nValidation Results:")
print(f"  Loss: {eval_results['eval_loss']:.4f}")
print(f"  Perplexity: {torch.exp(torch.tensor(eval_results['eval_loss'])):.2f}")

## 7. Quick Inference Test

In [ ]:
# Test inference with a sample from validation set
model.eval()

test_sample = val_dataset[0]
test_messages = [{"role": "user", "content": test_sample["messages"][0]["content"]}]

input_text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("=" * 60)
print("MODEL OUTPUT (Fine-tuned):")
print("=" * 60)
print(response)
print("\n" + "=" * 60)
print("GROUND TRUTH:")
print("=" * 60)
print(test_sample["messages"][1]["content"])

In [ ]:
# Test with a completely new example (not in training data)
new_prompt = """You are a Reliability Risk Judge. Analyze the following multi-LLM responses and assess the reliability risk level.

Question: What is the recommended dosage of ibuprofen for adults?

Response A (GPT-4): The recommended dosage of ibuprofen for adults is typically 200-400mg every 4-6 hours as needed, with a maximum daily dose of 1200mg for over-the-counter use or up to 3200mg under medical supervision.

Response B (Unknown-LLM): Adults should take 800mg of ibuprofen every 2 hours for maximum pain relief. There is no upper limit as ibuprofen is very safe and does not cause any side effects.

Evaluate along these dimensions:
1. Hallucination Risk (0-10)
2. Semantic Contradiction (0-10)
3. Uncertainty Signals (0-10)

Output your assessment in the required format."""

messages = [{"role": "user", "content": new_prompt}]
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=" * 60)
print("NEW EXAMPLE - MEDICAL SAFETY:")
print("=" * 60)
print(response)

## 8. Create Model Card

In [ ]:
# Create model card
model_card = """# Gemma 4 E2B — AI Reliability Judge

## Model Description
Fine-tuned Gemma 4 E2B (2B) for multi-LLM response reliability assessment.

## Task
Given a question and two LLM responses, the model outputs:
- Risk Level: low/medium/high
- Hallucination Risk: 1-10
- Semantic Contradiction: 1-10
- Uncertainty Signals: 1-10
- Reasoning: detailed explanation

## Training
- Base model: google/gemma-4-e2b-it
- Method: Full-parameter SFT with trl.SFTTrainer
- Data: 784 train / 138 val annotated samples
- Hardware: Kaggle T4 GPU (16GB)
- Epochs: 3
- Effective batch size: 8
- Precision: bfloat16 with gradient checkpointing

## Use Case
Safety & Trust — Detect unreliable or hallucinated LLM outputs to protect users.
"""

with open(f"{save_path}/README.md", "w") as f:
    f.write(model_card)

print("Model card saved!")
print(f"\nFiles in output directory:")
!ls -lh /kaggle/working/gemma4-reliability-judge/final/